In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score


In [2]:
# -----------------------------------
# 1 Load Diabetes Dataset
# -----------------------------------

data = load_diabetes()

X = data.data
y = data.target

# Convert regression target to classification
y = (y > np.mean(y)).astype(int)

In [3]:
# -----------------------------------
# 2 Normalize Features
# -----------------------------------

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

In [4]:
# -----------------------------------
# 3 Combine Data
# -----------------------------------

dataset = np.column_stack((X, y))

In [6]:
# -----------------------------------
# 4 Split Data Across Clients
# -----------------------------------

num_clients = 3

np.random.shuffle(dataset)

clients = np.array_split(dataset, num_clients)

In [7]:
# -----------------------------------
# 5 Initialize Global Model
# -----------------------------------

num_features = X.shape[1]

global_W = np.zeros(num_features)
global_b = 0

In [8]:
# -----------------------------------
# 6 Sigmoid Function
# -----------------------------------

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [9]:
# -----------------------------------
# 7 Local Training
# -----------------------------------

def local_train(client_data, W, b, lr=0.05, epochs=20):

    X_local = client_data[:, :-1]
    y_local = client_data[:, -1]

    W_local = W.copy()
    b_local = b

    n = len(y_local)

    for _ in range(epochs):

        z = np.dot(X_local, W_local) + b_local
        pred = sigmoid(z)

        error = pred - y_local

        grad_W = np.dot(X_local.T, error) / n
        grad_b = np.mean(error)

        W_local -= lr * grad_W
        b_local -= lr * grad_b

    return W_local, b_local, n

In [10]:
# -----------------------------------
# 8 Federated Averaging
# -----------------------------------

def fedavg(updates):

    total_samples = sum(size for _, _, size in updates)

    new_W = np.zeros_like(updates[0][0])
    new_b = 0

    for W, b, size in updates:

        weight = size / total_samples

        new_W += weight * W
        new_b += weight * b

    return new_W, new_b

In [11]:
# -----------------------------------
# 9 Federated Training
# -----------------------------------

rounds = 10

for r in range(rounds):

    updates = []

    for client in clients:

        W_local, b_local, size = local_train(client, global_W, global_b)

        updates.append((W_local, b_local, size))

    global_W, global_b = fedavg(updates)

    print(f"Round {r+1} completed")

Round 1 completed
Round 2 completed
Round 3 completed
Round 4 completed
Round 5 completed
Round 6 completed
Round 7 completed
Round 8 completed
Round 9 completed
Round 10 completed


In [12]:
# -----------------------------------
# 10 Evaluate Global Model
# -----------------------------------

X_all = dataset[:, :-1]
y_all = dataset[:, -1]

z = np.dot(X_all, global_W) + global_b
pred = sigmoid(z)

pred_class = (pred > 0.5).astype(int)

acc = accuracy_score(y_all, pred_class)

print("\nFinal Global Model Accuracy:", acc)


Final Global Model Accuracy: 0.7194570135746606
